# Data Formats & Storage — 02: Apache Iceberg Advanced

## What you will learn
Apache Iceberg is a **table format** — not a storage engine, not a file format.
It sits on top of Parquet/ORC/Avro and adds:
- **ACID transactions** on data lakes (no more corrupted tables from concurrent writes)
- **Time travel** — query your data as it looked yesterday, last week, or any snapshot
- **Schema evolution** — add, rename, drop columns without rewriting data
- **Partition evolution** — change partitioning strategy without migrating data
- **Hidden partitioning** — no more `WHERE dt=2024-01-01` partition filters in SQL
- **Row-level operations** — `MERGE INTO`, `UPDATE`, `DELETE` on Parquet files

In this notebook you will work through all of these features step by step,
understanding both **what** Iceberg does and **how** it works internally.

## How Iceberg Works Internally
```
Your Table (logical view)
        │
        ▼
  Metadata Layer  ← Iceberg manages this
  ┌─────────────────────────────────────────────┐
  │  metadata/                                  │
  │    v1.metadata.json  ← snapshot 1           │
  │    v2.metadata.json  ← snapshot 2 (current) │
  │    snap-001.avro     ← manifest list        │
  │    manifest-001.avro ← list of data files   │
  └─────────────────────────────────────────────┘
        │
        ▼
  Data Layer  ← actual Parquet files
  ┌─────────────────────────────────────────────┐
  │  data/                                      │
  │    part-00001.parquet                        │
  │    part-00002.parquet                        │
  │    ...                                      │
  └─────────────────────────────────────────────┘

Every write creates a NEW SNAPSHOT.
Old snapshots remain → time travel is free.
Atomic pointer swap → readers always see consistent state.
```


In [1]:
import os, time, datetime, pathlib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'
MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'
WAREHOUSE = f'{DATA_DIR}/warehouse/iceberg'

spark = (
    SparkSession.builder
    .appName('iceberg-advanced')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    # Iceberg catalog already configured in spark-defaults.conf:
    #   spark.sql.extensions = IcebergSparkSessionExtensions
    #   spark.sql.catalog.local = org.apache.iceberg.spark.SparkCatalog
    #   spark.sql.catalog.local.type = hadoop
    #   spark.sql.catalog.local.warehouse = /workspace/data/warehouse/iceberg
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Gluten: {GLUTEN_ENABLED}")
print(f"Iceberg warehouse: {WAREHOUSE}")

Spark 4.0.2 | Gluten: False
Iceberg warehouse: /workspace/data/warehouse/iceberg


## Step 1 — Create Your First Iceberg Table

We create a realistic `orders` table that we will evolve throughout this notebook.
Notice we use the `local` catalog (configured in spark-defaults.conf) with
the syntax `local.db.table_name`.


In [2]:
# Clean slate
spark.sql("CREATE DATABASE IF NOT EXISTS local.shop")
spark.sql("DROP TABLE IF EXISTS local.shop.orders")

# Create Iceberg table with explicit schema and partitioning
spark.sql("""
    CREATE TABLE local.shop.orders (
        order_id     BIGINT,
        customer_id  BIGINT,
        product      STRING,
        category     STRING,
        quantity     INT,
        unit_price   DOUBLE,
        total_amount DOUBLE,
        status       STRING,
        country      STRING,
        order_date   DATE,
        updated_at   TIMESTAMP
    )
    USING iceberg
    PARTITIONED BY (
        months(order_date),   -- hidden partition: extract month from order_date
        category              -- explicit partition
    )
    TBLPROPERTIES (
        'write.format.default'          = 'parquet',
        'write.parquet.compression-codec' = 'zstd',
        'history.expire.min-snapshots-to-keep' = '5'
    )
""")

print("Table created. Properties:")
spark.sql("DESCRIBE EXTENDED local.shop.orders").show(40, truncate=False)

Table created. Properties:
+----------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                                                                                  |comment|
+----------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|order_id                    |bigint                                                                                                                                                                     |NULL   |
|customer_id                 |bigint                                                                                             

## Step 2 — Insert Initial Data (Snapshot 1)

Every write to an Iceberg table creates a new **snapshot**.
A snapshot is an immutable point-in-time view of the table.


In [3]:
import random, datetime
random.seed(42)

PRODUCTS  = {"Laptop": ("Electronics", 999.99), "Phone": ("Electronics", 599.99),
             "Desk":   ("Furniture",   349.99),  "Chair": ("Furniture",   199.99),
             "Shirt":  ("Clothing",     49.99),   "Jeans": ("Clothing",    79.99),
             "Python Book": ("Books",   39.99),   "Spark Book": ("Books",   49.99)}
STATUSES  = ["pending", "confirmed", "shipped", "delivered", "cancelled"]
COUNTRIES = ["US", "UK", "DE", "FR", "JP", "CA"]

def make_orders(n, start_id=1, date_range=("2024-01-01","2024-03-31"), status_weights=None):
    rows = []
    start = datetime.date.fromisoformat(date_range[0])
    end   = datetime.date.fromisoformat(date_range[1])
    days  = (end - start).days
    for i in range(n):
        prod, (cat, price) = random.choice(list(PRODUCTS.items()))
        qty    = random.randint(1, 5)
        total  = round(qty * price * random.uniform(0.9, 1.1), 2)
        status = random.choice(status_weights or STATUSES)
        odate  = start + datetime.timedelta(days=random.randint(0, days))
        rows.append((start_id + i, random.randint(1, 200), prod, cat,
                     qty, price, total, status, random.choice(COUNTRIES),
                     odate, datetime.datetime.now()))
    return rows

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("quantity",    IntegerType()),
    StructField("unit_price",  DoubleType()),
    StructField("total_amount",DoubleType()),
    StructField("status",      StringType()),
    StructField("country",     StringType()),
    StructField("order_date",  DateType()),
    StructField("updated_at",  TimestampType()),
])

# Insert batch 1: Q1 2024 orders
batch1 = spark.createDataFrame(make_orders(500, start_id=1), schema)
batch1.writeTo("local.shop.orders").append()

print(f"Inserted {batch1.count()} orders (Q1 2024)")
print(f"Total rows: {spark.table('local.shop.orders').count()}")

Inserted 500 orders (Q1 2024)
Total rows: 500


In [4]:
# Check the snapshot that was created
spark.sql("""
    SELECT snapshot_id, committed_at, operation,
           summary['added-records']    AS added_records,
           summary['total-records']    AS total_records,
           summary['added-files-size'] AS added_bytes
    FROM local.shop.orders.snapshots
""").show(truncate=False)

+-------------------+-----------------------+---------+-------------+-------------+-----------+
|snapshot_id        |committed_at           |operation|added_records|total_records|added_bytes|
+-------------------+-----------------------+---------+-------------+-------------+-----------+
|1713857722965304523|2026-04-16 17:52:36.378|append   |500          |500          |51723      |
+-------------------+-----------------------+---------+-------------+-------------+-----------+



## Step 3 — MERGE INTO: Upsert Pattern

`MERGE INTO` is the most powerful Iceberg operation.
It handles **insert + update + delete** in a single atomic operation.

**Use case:** A nightly sync from your OLTP database — some orders have been updated
(status changed), some are new. You want to apply all changes atomically.


In [5]:
# Simulate: 100 existing orders got status updates + 100 new orders arrived
# This is a classic CDC (Change Data Capture) pattern

updates_and_inserts = (
    # 100 existing orders: status updated to delivered
    spark.createDataFrame(
        [(i, "delivered", datetime.datetime.now())
         for i in random.sample(range(1, 501), 100)],
        ["order_id", "new_status", "updated_at"]
    )
)

# 100 new orders (Q2 2024)
batch2 = spark.createDataFrame(make_orders(100, start_id=501,
              date_range=("2024-04-01","2024-06-30")), schema)
batch2.createOrReplaceTempView("new_orders")
updates_and_inserts.createOrReplaceTempView("status_updates")

print("Before MERGE:")
spark.sql("""
    SELECT status, COUNT(*) as cnt
    FROM local.shop.orders
    GROUP BY status ORDER BY cnt DESC
""").show()

# MERGE: update existing + insert new
spark.sql("""
    MERGE INTO local.shop.orders t
    USING (
        SELECT o.order_id, o.customer_id, o.product, o.category,
               o.quantity, o.unit_price, o.total_amount,
               COALESCE(u.new_status, o.status) AS status,
               o.country, o.order_date,
               COALESCE(u.updated_at, o.updated_at) AS updated_at
        FROM new_orders o
        LEFT JOIN status_updates u ON o.order_id = u.order_id
        UNION ALL
        SELECT o.order_id, o.customer_id, o.product, o.category,
               o.quantity, o.unit_price, o.total_amount,
               u.new_status AS status,
               o.country, o.order_date, u.updated_at
        FROM local.shop.orders o
        JOIN status_updates u ON o.order_id = u.order_id
    ) s
    ON t.order_id = s.order_id
    WHEN MATCHED THEN
        UPDATE SET t.status = s.status, t.updated_at = s.updated_at
    WHEN NOT MATCHED THEN
        INSERT *
""")

print("\nAfter MERGE:")
spark.sql("""
    SELECT status, COUNT(*) as cnt
    FROM local.shop.orders
    GROUP BY status ORDER BY cnt DESC
""").show()

total = spark.table("local.shop.orders").count()
print(f"Total rows: {total} (was 500, added 100 new → 600)")

Before MERGE:


+---------+---+
|   status|cnt|
+---------+---+
|  pending|111|
|  shipped|104|
|delivered|101|
|confirmed| 97|
|cancelled| 87|
+---------+---+




After MERGE:
+---------+---+
|   status|cnt|
+---------+---+
|delivered|200|
|  pending|114|
|  shipped|100|
|confirmed| 99|
|cancelled| 87|
+---------+---+

Total rows: 600 (was 500, added 100 new → 600)


## Step 4 — Time Travel

Because Iceberg keeps all snapshots, you can **query the table as it was at any point in time**.

This is invaluable for:
- Debugging: "what did the data look like before that bad pipeline run?"
- Auditing: "what was the order status on January 15th?"
- Reproducibility: "re-run the ML model on the exact data from last quarter"
- Rolling back mistakes: "oops, that DELETE was wrong"


In [6]:
# Get all snapshots
snapshots = spark.sql("""
    SELECT snapshot_id, committed_at, operation,
           summary['added-records']  AS added,
           summary['deleted-records'] AS deleted,
           summary['total-records']  AS total
    FROM local.shop.orders.snapshots
    ORDER BY committed_at
""")
snapshots.show(truncate=False)

# Collect snapshot IDs for time travel
snap_rows = snapshots.collect()
snap1_id = snap_rows[0]["snapshot_id"]
snap2_id = snap_rows[-1]["snapshot_id"]  # latest

print(f"Snapshot 1 (initial insert): {snap1_id}")
print(f"Snapshot 2 (after MERGE)   : {snap2_id}")

+-------------------+-----------------------+---------+-----+-------+-----+
|snapshot_id        |committed_at           |operation|added|deleted|total|
+-------------------+-----------------------+---------+-----+-------+-----+
|1713857722965304523|2026-04-16 17:52:36.378|append   |500  |NULL   |500  |
|274793347457729903 |2026-04-16 17:52:46.443|overwrite|600  |500    |600  |
+-------------------+-----------------------+---------+-----+-------+-----+

Snapshot 1 (initial insert): 1713857722965304523
Snapshot 2 (after MERGE)   : 274793347457729903


In [7]:
# Time travel by snapshot ID
print("== Snapshot 1 (original 500 orders) ==")
spark.sql(f"""
    SELECT status, COUNT(*) as cnt
    FROM local.shop.orders VERSION AS OF {snap1_id}
    GROUP BY status ORDER BY cnt DESC
""").show()

print("== Snapshot 2 (current: 600 orders with updates) ==")
spark.sql(f"""
    SELECT status, COUNT(*) as cnt
    FROM local.shop.orders VERSION AS OF {snap2_id}
    GROUP BY status ORDER BY cnt DESC
""").show()

# Time travel by timestamp (as of 1 second ago)
as_of = (datetime.datetime.now() - datetime.timedelta(seconds=1)).strftime("%Y-%m-%d %H:%M:%S")
print(f"== As of timestamp {as_of} ==")
spark.sql(f"""
    SELECT COUNT(*) as total_orders
    FROM local.shop.orders TIMESTAMP AS OF '{as_of}'
""").show()

== Snapshot 1 (original 500 orders) ==
+---------+---+
|   status|cnt|
+---------+---+
|  pending|111|
|  shipped|104|
|delivered|101|
|confirmed| 97|
|cancelled| 87|
+---------+---+

== Snapshot 2 (current: 600 orders with updates) ==
+---------+---+
|   status|cnt|
+---------+---+
|delivered|200|
|  pending|114|
|  shipped|100|
|confirmed| 99|
|cancelled| 87|
+---------+---+

== As of timestamp 2026-04-16 17:52:48 ==
+------------+
|total_orders|
+------------+
|         600|
+------------+



In [8]:
# Compare specific order across snapshots
sample_order_id = 50  # an order that was updated

print(f"Order {sample_order_id} — before MERGE (snapshot 1):")
spark.sql(f"""
    SELECT order_id, status, updated_at
    FROM local.shop.orders VERSION AS OF {snap1_id}
    WHERE order_id = {sample_order_id}
""").show(truncate=False)

print(f"Order {sample_order_id} — after MERGE (current):")
spark.sql(f"""
    SELECT order_id, status, updated_at
    FROM local.shop.orders
    WHERE order_id = {sample_order_id}
""").show(truncate=False)

Order 50 — before MERGE (snapshot 1):
+--------+---------+--------------------------+
|order_id|status   |updated_at                |
+--------+---------+--------------------------+
|50      |confirmed|2026-04-16 17:52:30.902551|
+--------+---------+--------------------------+

Order 50 — after MERGE (current):
+--------+---------+--------------------------+
|order_id|status   |updated_at                |
+--------+---------+--------------------------+
|50      |confirmed|2026-04-16 17:52:30.902551|
+--------+---------+--------------------------+



## Step 5 — Schema Evolution

Traditional data lakes require rewriting all data to change the schema.
Iceberg handles schema changes **metadata-only** — no data rewrite needed.

Supported operations:
- `ADD COLUMN` — add new column (reads as NULL for existing rows)
- `DROP COLUMN` — hide column (data still in files, just not exposed)
- `RENAME COLUMN` — rename without rewrite
- `ALTER COLUMN` — change type (within safe promotions: int→long, float→double)
- `REORDER COLUMNS` — cosmetic, no rewrite


In [9]:
print("Current schema:")
spark.sql("DESCRIBE local.shop.orders").show(truncate=False)

# Add a discount column and a loyalty_points column
spark.sql("""
    ALTER TABLE local.shop.orders
    ADD COLUMNS (
        discount_pct  DOUBLE COMMENT 'Discount percentage applied 0.0-1.0',
        loyalty_points INT   COMMENT 'Loyalty points earned for this order',
        is_gift       BOOLEAN COMMENT 'Whether this is a gift order'
    )
""")

print("\nSchema after ADD COLUMNS:")
spark.sql("DESCRIBE local.shop.orders").show(truncate=False)

Current schema:
+--------------+------------------+-------+
|col_name      |data_type         |comment|
+--------------+------------------+-------+
|order_id      |bigint            |NULL   |
|customer_id   |bigint            |NULL   |
|product       |string            |NULL   |
|category      |string            |NULL   |
|quantity      |int               |NULL   |
|unit_price    |double            |NULL   |
|total_amount  |double            |NULL   |
|status        |string            |NULL   |
|country       |string            |NULL   |
|order_date    |date              |NULL   |
|updated_at    |timestamp         |NULL   |
|              |                  |       |
|# Partitioning|                  |       |
|Part 0        |months(order_date)|       |
|Part 1        |category          |       |
+--------------+------------------+-------+


Schema after ADD COLUMNS:
+--------------+------------------+------------------------------------+
|col_name      |data_type         |comment     

In [10]:
# Existing rows see NULL for new columns — no data rewrite!
print("Existing rows — new columns appear as NULL:")
spark.sql("""
    SELECT order_id, total_amount, discount_pct, loyalty_points, is_gift
    FROM local.shop.orders
    LIMIT 5
""").show()

# New inserts can populate the new columns
batch3 = spark.createDataFrame(
    make_orders(50, start_id=601, date_range=("2024-07-01","2024-09-30")),
    schema  # original schema
)
# Add the new columns
batch3_enriched = batch3 \
    .withColumn("discount_pct",   F.round(F.rand(42) * 0.3, 2)) \
    .withColumn("loyalty_points", (F.col("total_amount") * 10).cast("int")) \
    .withColumn("is_gift",        F.rand(42) > 0.85)

batch3_enriched.writeTo("local.shop.orders").append()

print("\nNew rows with populated columns:")
spark.sql("""
    SELECT order_id, total_amount, discount_pct, loyalty_points, is_gift
    FROM local.shop.orders
    WHERE order_id >= 601
    LIMIT 5
""").show()

# Rename a column
spark.sql("ALTER TABLE local.shop.orders RENAME COLUMN is_gift TO gift_order")
print("\nColumn renamed: is_gift → gift_order")
spark.sql("DESCRIBE local.shop.orders").filter(F.col("col_name").isin(
    ["discount_pct","loyalty_points","gift_order"])).show()

Existing rows — new columns appear as NULL:
+--------+------------+------------+--------------+-------+
|order_id|total_amount|discount_pct|loyalty_points|is_gift|
+--------+------------+------------+--------------+-------+
|       9|       53.54|        NULL|          NULL|   NULL|
|      16|      246.37|        NULL|          NULL|   NULL|
|      50|       54.38|        NULL|          NULL|   NULL|
|      59|       339.3|        NULL|          NULL|   NULL|
|      64|       46.07|        NULL|          NULL|   NULL|
+--------+------------+------------+--------------+-------+




New rows with populated columns:
+--------+------------+------------+--------------+-------+
|order_id|total_amount|discount_pct|loyalty_points|is_gift|
+--------+------------+------------+--------------+-------+
|     613|     4722.66|        0.24|         47226|  false|
|     625|     5435.25|        0.24|         54352|  false|
|     646|      377.55|        0.09|          3775|  false|
|     629|      267.98|        0.22|          2679|  false|
|     635|      139.36|        0.29|          1393|   true|
+--------+------------+------------+--------------+-------+


Column renamed: is_gift → gift_order
+--------------+---------+--------------------+
|      col_name|data_type|             comment|
+--------------+---------+--------------------+
|  discount_pct|   double|Discount percenta...|
|loyalty_points|      int|Loyalty points ea...|
|    gift_order|  boolean|Whether this is a...|
+--------------+---------+--------------------+



## Step 6 — Partition Evolution

This is one of Iceberg's killer features.

In traditional Hive-partitioned tables, if you want to change the partitioning strategy
(e.g., from daily to monthly), you must **rewrite ALL your data**. That can be petabytes.

With Iceberg, you just change the partition spec. **Old data keeps its old layout,
new data uses the new layout.** Iceberg handles reading both transparently.


In [11]:
# Show current partitioning
spark.sql("SELECT * FROM local.shop.orders.partitions LIMIT 10").show(truncate=False)

print("Current partition spec (months + category):")
print("  partition 1: months(order_date)")
print("  partition 2: category")
print()

# Evolve: replace monthly partitioning with daily for recent data
# (useful when data volume increased and monthly buckets got too large)
spark.sql("""
    ALTER TABLE local.shop.orders
    REPLACE PARTITION FIELD months(order_date)
    WITH days(order_date)
""")

print("Partition spec evolved to: days(order_date) + category")
print("Old data (monthly partitioned) is NOT rewritten.")
print("New data will use daily partitioning.")
print()

# Insert new data — it uses the NEW daily partition scheme
batch4 = spark.createDataFrame(
    make_orders(50, start_id=651, date_range=("2024-10-01","2024-10-31")),
    schema
)
batch4_enriched = batch4 \
    .withColumn("discount_pct",   F.lit(None).cast("double")) \
    .withColumn("loyalty_points", F.lit(None).cast("int")) \
    .withColumn("gift_order",     F.lit(None).cast("boolean"))

batch4_enriched.writeTo("local.shop.orders").append()

print("Inserted October 2024 data with daily partitioning")
print()

# Iceberg reads both old (monthly) and new (daily) partitions seamlessly
result = spark.sql("""
    SELECT
        YEAR(order_date)  AS year,
        MONTH(order_date) AS month,
        COUNT(*)          AS orders,
        SUM(total_amount) AS revenue
    FROM local.shop.orders
    GROUP BY 1, 2
    ORDER BY 1, 2
""")
result.show()
print(f"Total orders across all partition formats: {spark.table('local.shop.orders').count()}")

+------------------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|partition         |spec_id|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|last_updated_at        |last_updated_snapshot_id|
+------------------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|{649, Clothing}   |0      |36          |1         |4252                         |0                           |0                         |0                           |0                         |2026-04-16 17:52:46.443|274793347457729903      |
|{656, Books}      |0   

Inserted October 2024 data with daily partitioning



[Stage 81:===============================================>        (16 + 3) / 19]

+----+-----+------+------------------+
|year|month|orders|           revenue|
+----+-----+------+------------------+
|2024|    1|   181|163388.32999999996|
|2024|    2|   173|151357.08000000002|
|2024|    3|   146|111489.59999999999|
|2024|    4|    25|           16068.5|
|2024|    5|    30|29526.579999999998|
|2024|    6|    45| 38603.35999999999|
|2024|    7|    21|          13935.64|
|2024|    8|     9|          13402.98|
|2024|    9|    20|          23661.65|
|2024|   10|    50|45070.380000000005|
+----+-----+------+------------------+

Total orders across all partition formats: 700


## Step 7 — Row-Level Deletes (DELETE + UPDATE)

Iceberg supports row-level `DELETE` and `UPDATE` statements using **deletion vectors**
or **copy-on-write** (depending on configuration).

This is essential for:
- GDPR/right-to-be-forgotten: `DELETE FROM orders WHERE customer_id = 12345`
- Correcting bad data: `UPDATE orders SET status='cancelled' WHERE ...`
- Data masking pipelines


In [12]:
print("Before DELETE — cancelled orders:")
spark.sql("""
    SELECT COUNT(*) as cancelled_count
    FROM local.shop.orders
    WHERE status = 'cancelled'
""").show()

# DELETE: remove all cancelled orders older than Q1 2024
spark.sql("""
    DELETE FROM local.shop.orders
    WHERE status = 'cancelled'
    AND order_date < '2024-04-01'
""")

print("After DELETE — cancelled orders from Q1 removed:")
spark.sql("""
    SELECT status, COUNT(*) as cnt
    FROM local.shop.orders
    GROUP BY status ORDER BY cnt DESC
""").show()

# UPDATE: apply 10% discount to all Electronics orders in October
spark.sql("""
    UPDATE local.shop.orders
    SET discount_pct = 0.10,
        total_amount = total_amount * 0.90,
        updated_at   = current_timestamp()
    WHERE category = 'Electronics'
    AND   order_date >= '2024-10-01'
""")

print("After UPDATE — Electronics October orders discounted:")
spark.sql("""
    SELECT product, COUNT(*) as orders,
           AVG(discount_pct) as avg_discount,
           SUM(total_amount) as revenue
    FROM local.shop.orders
    WHERE category = 'Electronics' AND order_date >= '2024-10-01'
    GROUP BY product
""").show()

Before DELETE — cancelled orders:


+---------------+
|cancelled_count|
+---------------+
|            115|
+---------------+



After DELETE — cancelled orders from Q1 removed:


+---------+---+
|   status|cnt|
+---------+---+
|delivered|217|
|  pending|130|
|confirmed|121|
|  shipped|117|
|cancelled| 43|
+---------+---+



After UPDATE — Electronics October orders discounted:
+-------+------+-------------------+---------+
|product|orders|       avg_discount|  revenue|
+-------+------+-------------------+---------+
| Laptop|     6|0.10000000000000002|19729.674|
|  Phone|     8|                0.1| 9185.544|
+-------+------+-------------------+---------+



## Step 8 — Snapshot Management & Table Maintenance

Iceberg accumulates snapshots. Without maintenance:
- Old snapshot metadata grows indefinitely
- Old data files (referenced only by expired snapshots) waste disk space

Iceberg provides:
- `expireSnapshots` — remove old snapshot metadata
- `deleteOrphanFiles` — clean up unreferenced data files
- `rewriteDataFiles` — compact small files into larger ones


In [13]:
# Show all snapshots before maintenance
print("Snapshots before maintenance:")
spark.sql("""
    SELECT snapshot_id,
           committed_at,
           operation,
           summary['total-records'] AS total_records
    FROM local.shop.orders.snapshots
    ORDER BY committed_at
""").show(truncate=False)

snap_count_before = spark.sql("SELECT COUNT(*) FROM local.shop.orders.snapshots").collect()[0][0]
print(f"Total snapshots: {snap_count_before}")

Snapshots before maintenance:
+-------------------+-----------------------+---------+-------------+
|snapshot_id        |committed_at           |operation|total_records|
+-------------------+-----------------------+---------+-------------+
|1713857722965304523|2026-04-16 17:52:36.378|append   |500          |
|274793347457729903 |2026-04-16 17:52:46.443|overwrite|600          |
|5178451999904236236|2026-04-16 17:52:52.928|append   |650          |
|5571416416233866820|2026-04-16 17:52:59.002|append   |700          |
|202713797316543399 |2026-04-16 17:53:23.113|overwrite|628          |
|7254387244684354349|2026-04-16 17:53:28.798|overwrite|628          |
+-------------------+-----------------------+---------+-------------+

Total snapshots: 6


In [14]:
from pyspark.sql import SparkSession

# Expire snapshots older than current time minus 0 seconds
# (in production you'd keep last 7 days: retain_last=7 * 86400 * 1000 ms)
# Here we keep the latest 2 snapshots for demo purposes

latest_snapshots = spark.sql("""
    SELECT snapshot_id FROM local.shop.orders.snapshots
    ORDER BY committed_at DESC LIMIT 2
""").collect()
keep_ids = {r["snapshot_id"] for r in latest_snapshots}

# Use Iceberg's expire_snapshots procedure
spark.sql("""
    CALL local.system.expire_snapshots(
        table => 'local.shop.orders',
        retain_last => 2
    )
""").show()

print("\nSnapshots after expiry (keeping last 2):")
spark.sql("""
    SELECT snapshot_id, committed_at, operation,
           summary['total-records'] AS records
    FROM local.shop.orders.snapshots
    ORDER BY committed_at
""").show(truncate=False)

+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|deleted_data_files_count|deleted_position_delete_files_count|deleted_equality_delete_files_count|deleted_manifest_files_count|deleted_manifest_lists_count|deleted_statistics_files_count|
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|                       0|                                  0|                                  0|                           0|                           0|                             0|
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+


Snapshots after expiry (keeping last 2):
+----------------

In [15]:
# Rewrite data files: compact many small files into fewer large ones
# This is important after many small inserts or row-level deletes
print("Data files before rewrite:")
spark.sql("""
    SELECT COUNT(*) as file_count,
           SUM(file_size_in_bytes) / 1024 / 1024 AS total_mb,
           AVG(file_size_in_bytes) / 1024        AS avg_kb_per_file
    FROM local.shop.orders.files
""").show()

# Compact to target file size of 128MB
spark.sql("""
    CALL local.system.rewrite_data_files(
        table            => 'local.shop.orders',
        strategy         => 'binpack',
        options          => map('target-file-size-bytes', '134217728',
                                'min-file-size-bytes',    '67108864')
    )
""").show()

print("Data files after rewrite:")
spark.sql("""
    SELECT COUNT(*) as file_count,
           SUM(file_size_in_bytes) / 1024 / 1024 AS total_mb,
           AVG(file_size_in_bytes) / 1024        AS avg_kb_per_file
    FROM local.shop.orders.files
""").show()

Data files before rewrite:
+----------+------------------+------------------+
|file_count|          total_mb|   avg_kb_per_file|
+----------+------------------+------------------+
|       322|1.2341251373291016|3.9246712441770186|
+----------+------------------+------------------+



+--------------------------+----------------------+---------------------+-----------------------+--------------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|removed_delete_files_count|
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+
|                        24|                   135|                95260|                      0|                         0|
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+

Data files after rewrite:
+----------+------------------+-----------------+
|file_count|          total_mb|  avg_kb_per_file|
+----------+------------------+-----------------+
|       433|1.6572961807250977|3.919333231091224|
+----------+------------------+-----------------+



## Step 9 — Table History & Audit Log

Iceberg keeps a complete audit trail of every operation.
This is invaluable for data governance and debugging.


In [16]:
# Full table history
print("Complete table history:")
spark.sql("""
    SELECT made_current_at,
           snapshot_id,
           parent_id,
           is_current_ancestor
    FROM local.shop.orders.history
    ORDER BY made_current_at
""").show(truncate=False)

# What changed between two snapshots?
snaps = spark.sql("""
    SELECT snapshot_id FROM local.shop.orders.snapshots
    ORDER BY committed_at DESC LIMIT 2
""").collect()

if len(snaps) >= 2:
    newer = snaps[0]["snapshot_id"]
    older = snaps[1]["snapshot_id"]
    print(f"\nFiles changed between snapshot {older} and {newer}:")
    spark.sql(f"""
        SELECT file_path, content, file_size_in_bytes,
               record_count, partition
        FROM local.shop.orders.files
        LIMIT 5
    """).show(truncate=False)

# Table properties and statistics
print("\nTable statistics:")
spark.sql("""
    SELECT COUNT(*)          AS total_rows,
           COUNT(DISTINCT category) AS categories,
           MIN(order_date)   AS earliest_order,
           MAX(order_date)   AS latest_order,
           SUM(total_amount) AS lifetime_revenue
    FROM local.shop.orders
""").show()

Complete table history:
+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-04-16 17:52:36.378|1713857722965304523|NULL               |true               |
|2026-04-16 17:52:46.443|274793347457729903 |1713857722965304523|true               |
|2026-04-16 17:52:52.928|5178451999904236236|274793347457729903 |true               |
|2026-04-16 17:52:59.002|5571416416233866820|5178451999904236236|true               |
|2026-04-16 17:53:23.113|202713797316543399 |5571416416233866820|true               |
|2026-04-16 17:53:28.798|7254387244684354349|202713797316543399 |true               |
|2026-04-16 17:53:39.272|171396475462258406 |7254387244684354349|true               |
+-----------------------+-------------------+-------------------+-------------------+


Files changed between snapsh

[Stage 132:===================================================> (105 + 4) / 109]

+----------+----------+--------------+------------+----------------+
|total_rows|categories|earliest_order|latest_order|lifetime_revenue|
+----------+----------+--------------+------------+----------------+
|       628|         4|    2024-01-01|  2024-10-31|      538890.508|
+----------+----------+--------------+------------+----------------+



## Step 10 — Branching and Tagging (Iceberg 1.1+)

Iceberg supports **branches** and **tags** — like Git, but for your data.

- **Tags**: named snapshots for reproducibility (`tag: 'end_of_q1_2024'`)
- **Branches**: isolated write paths for testing pipelines without affecting production

**Use cases:**
- Tag your data before a major pipeline run → easy rollback if it goes wrong
- Create a `staging` branch → test transformations on real data → merge if correct
- Audit: tag snapshots for regulatory compliance (`'sox_q3_2024'`)


In [17]:
# Create a tag for the current snapshot (end of our demo data)
spark.sql("""
    ALTER TABLE local.shop.orders
    CREATE TAG end_of_demo
    RETAIN 30 DAYS
""")

# Create a branch for experimental work
spark.sql("""
    ALTER TABLE local.shop.orders
    CREATE BRANCH experimental
    RETAIN 7 DAYS
""")

print("Tags and branches created:")
spark.sql("""
    SELECT name, type, snapshot_id, max_reference_age_in_ms
    FROM local.shop.orders.refs
""").show(truncate=False)

# Write to the experimental branch WITHOUT affecting main
experimental_data = spark.createDataFrame(
    make_orders(10, start_id=9001, date_range=("2025-01-01","2025-01-31")),
    schema
).withColumn("discount_pct",   F.lit(0.25)) \
 .withColumn("loyalty_points", F.lit(500)) \
 .withColumn("gift_order",     F.lit(False))

experimental_data.writeTo("local.shop.orders.branch_experimental").append()

print("\nMain branch row count:")
spark.table("local.shop.orders").count()

print("\nExperimental branch row count (includes 10 test rows):")
spark.sql("SELECT COUNT(*) FROM local.shop.orders VERSION AS OF 'experimental'").show()

# Query the tag (reproducible historical view)
print("\nQuerying via tag 'end_of_demo':")
spark.sql("""
    SELECT COUNT(*) as rows_at_tag
    FROM local.shop.orders VERSION AS OF 'end_of_demo'
""").show()

Tags and branches created:
+------------+------+------------------+-----------------------+
|name        |type  |snapshot_id       |max_reference_age_in_ms|
+------------+------+------------------+-----------------------+
|end_of_demo |TAG   |171396475462258406|2592000000             |
|main        |BRANCH|171396475462258406|NULL                   |
|experimental|BRANCH|171396475462258406|604800000              |
+------------+------+------------------+-----------------------+




Main branch row count:

Experimental branch row count (includes 10 test rows):
+--------+
|count(1)|
+--------+
|     638|
+--------+


Querying via tag 'end_of_demo':
+-----------+
|rows_at_tag|
+-----------+
|        628|
+-----------+



----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 35452)
Traceback (most recent call last):
  File "/usr/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/opt/spark/python/pyspark/accumulators.py", line 299, in handle
    poll(accum_updates)
  File "/opt/spark/python/pyspark/accumulators.py", line 271, in poll
    if self.rfile in r and func():
  File "/opt/spark/python/pyspark/accumulators.py", line 275, in accum_updates
    num_updates = read_int(self.rfile)
  File "/opt/spark/python/pyspark/serializers.py", lin

## Summary: Iceberg Feature Map

| Feature | SQL Syntax | Use case |
|---|---|---|
| Create table | `CREATE TABLE ... USING iceberg` | New table with ACID |
| Append | `INSERT INTO` / `writeTo().append()` | Incremental loads |
| Upsert | `MERGE INTO ... WHEN MATCHED ... WHEN NOT MATCHED` | CDC sync |
| Delete rows | `DELETE FROM ... WHERE` | GDPR, bad data |
| Update rows | `UPDATE ... SET ... WHERE` | Corrections |
| Time travel (snapshot) | `... VERSION AS OF 12345` | Reproducibility |
| Time travel (timestamp) | `... TIMESTAMP AS OF '2024-01-01'` | Point-in-time query |
| Add column | `ALTER TABLE ... ADD COLUMNS (...)` | Schema evolution |
| Rename column | `ALTER TABLE ... RENAME COLUMN` | Safe rename |
| Change partition | `ALTER TABLE ... REPLACE PARTITION FIELD` | Partition evolution |
| Expire snapshots | `CALL system.expire_snapshots(...)` | Storage cleanup |
| Compact files | `CALL system.rewrite_data_files(...)` | Small file problem |
| Create tag | `ALTER TABLE ... CREATE TAG name` | Audit / rollback point |
| Create branch | `ALTER TABLE ... CREATE BRANCH name` | Isolated experiments |

### When to use Iceberg vs Delta Lake
| Iceberg | Delta Lake |
|---|---|
| Multi-engine (Spark + Trino + Flink + Hive) | Primarily Spark ecosystem |
| Cloud-agnostic (S3, GCS, ADLS, HDFS) | Strong Azure + Databricks integration |
| Partition evolution without rewrite | OPTIMIZE + ZORDER for query acceleration |
| Branching/tagging built-in | Delta Sharing for cross-org sharing |
| Open spec (vendor-neutral) | Databricks-backed, widely adopted |

**Both are excellent choices.** Use Iceberg if you need multi-engine or cloud portability.
Use Delta if you're primarily Spark/Databricks.

**Next:** `03_delta_advanced.ipynb` — OPTIMIZE, ZORDER, VACUUM, Change Data Feed.
